![Banner](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/competitions/datacrunch-2/assets/banner.webp)

# DataCrunch 2 — Stacking Ensemble (sized for repeated `train()` calls)

**Why this version looks different from the last one:** Submission #2 was
terminated after 9h10min without finishing even the *first* of 4 base
models, in the *first* of **9 required `train()` calls** (the platform
retrains at every walk-forward step — `Train Frequency: 1`). DataCrunch's
weekly compute quota is ~10 hours total, shared across every run that
week, so a single `train()` call needs to take minutes, not hours.

**What changed to fit that budget:**
1. **Feature selection is now a single, fully vectorized Pearson-correlation
   filter** (`select_features_by_correlation`) instead of a `RandomForest`
   refit inside every CV fold. It runs a single matrix multiply over the
   whole training set — seconds, not hours, even at a few million rows —
   at the cost of missing nonlinear/interaction effects a model-based
   selector would catch. The base models below see the raw filtered
   features and can still learn interactions themselves.
2. **No hyperparameter search inside `train()`.** Searching per call was
   the other main cost driver, multiplied by 9. Instead, `XGB_FIXED_PARAMS`
   / `LGBM_FIXED_PARAMS` hold fixed, capped hyperparameters (bounded
   depth, moderate `n_estimators`) chosen to be fast and reasonable rather
   than exhaustively tuned. There's a separate, clearly-marked **offline
   tuning cell** near the bottom — run it manually against a subsample if
   you want to explore better values, then copy them into the fixed
   dicts. It isn't referenced by `train()`/`infer()`, so it's stripped
   from the actual submission automatically.
3. **Down to 2 base models (XGBoost, LightGBM) + Ridge meta-learner**,
   not 4. `ExtraTrees` and `LinearSVR` are dropped for now — not because
   they're bad ideas, but because every additional base model multiplies
   cost across all 9 `train()` calls, and we don't yet know the real
   per-call runtime at full scale. Once you've confirmed headroom under
   quota with this leaner version, they're easy to add back.
4. Base models now use `n_jobs=-1` directly (no more nested search
   wrapping them, so no oversubscription risk from doing so).
5. Still: `id`/`moon` dropped, every CV split (for the stack's internal
   out-of-fold predictions) grouped by `moon` via precomputed splits,
   scored on Spearman, `# @crunch/keep:on/off` markers around every
   global the runner needs, `crunch_tools.test()` before submitting.

**Please verify the ~10h/week figure and the exact walk-forward step count
on your competition's own resource page** — I'm going from DataCrunch's
general docs, and exact numbers can vary or change.


In [65]:
# Install the Crunch CLI + LightGBM
%pip install crunch-cli lightgbm --upgrade --quiet --progress-bar off

# Setup your local environment
!crunch setup-notebook datacrunch-2 8XGpZS4pf4K4uCNbykQC3cFc --size small

Note: you may need to restart the kernel to use updated packages.
zsh:1: /Users/Tia_987/Downloads/datacrunch/.venv/bin/crunch: bad interpreter: /Users/Tia_987/Downloads/AML25-main/.venv/bin/python3: no such file or directory


## Imports

In [66]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from lightgbm import LGBMRegressor  # >= 4.0
from scipy.stats import randint, uniform, spearmanr
from sklearn.ensemble import StackingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, BayesianRidge
from sklearn.metrics import make_scorer
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn import svm

warnings.filterwarnings("ignore")

In [67]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

loaded crunch tools for module: <module '__main__'>

cli version: 12.0.2
available ram: 16.00 gb
available cpu: 10 core
----


## Config & helpers

In [68]:
# @crunch/keep:on
RANDOM_STATE = 0
ID_COLUMNS = ["id", "moon"]
FEATURE_SELECTION_TOP_K = 300
# @crunch/keep:off


def get_feature_columns(df: pd.DataFrame):
    """All Feature_* columns — explicitly excludes id/moon/target so they
    never get fed into the model as if they were predictive features."""
    return [c for c in df.columns if c not in ID_COLUMNS and c != "target"]


def get_model_path(model_directory_path: str) -> str:
    return os.path.join(model_directory_path, "model.joblib")


def spearman(y_true, y_pred) -> float:
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(corr) else corr


# @crunch/keep:on
spearman_scorer = make_scorer(spearman, greater_is_better=True)
# @crunch/keep:off

### Data modifiers

In [69]:
def select_features_by_correlation(X: pd.DataFrame, y: np.ndarray, top_k: int = FEATURE_SELECTION_TOP_K):
    """Fast, fully vectorized Pearson-correlation feature filter — one
    matrix multiply over the whole training set. Replaces a RandomForest
    -based selector that couldn't finish inside the platform's runtime
    quota when refit per CV fold. Trade-off: linear-only, so nonlinear /
    interaction effects are left for the base models to find themselves."""
    X_vals = X.to_numpy(dtype=np.float32)
    y_vals = y.astype(np.float32)
    X_centered = X_vals - X_vals.mean(axis=0)
    y_centered = y_vals - y_vals.mean()
    numerator = X_centered.T @ y_centered
    denom = np.sqrt((X_centered ** 2).sum(axis=0) * (y_centered ** 2).sum()) + 1e-12
    corr = numerator / denom
    top_idx = np.argsort(-np.abs(corr))[:top_k]
    return [X.columns[i] for i in top_idx]

def select_features_by_information_ratio(X: pd.DataFrame, y: pd.Series, moons: pd.Series, top_k: int = FEATURE_SELECTION_TOP_K):
    """
    Replaces the global correlation filter with an Era-Wise Information Ratio filter.
    Instead of finding features that correlated well across the whole dataset (which is prone 
    to outlier eras), this finds features that consistently correlate across individual moons.
    """
    df = X.copy()
    df['target'] = y
    df['moon'] = moons
    
    # Calculate correlation for each feature against the target, partitioned by moon
    era_correlations = df.groupby('moon').corrwith(df['target'], method='pearson').drop(['target', 'moon'], axis=1, errors='ignore')
    
    # Calculate Information Ratio: Mean of era correlations / Std Dev of era correlations
    # A high IR means the feature is consistently predictive, regardless of market regime.
    mean_corr = era_correlations.mean()
    std_corr = era_correlations.std() + 1e-8 # Prevent division by zero
    info_ratio = (mean_corr.abs() / std_corr)
    
    # Sort and select top features
    top_features = info_ratio.sort_values(ascending=False).head(top_k).index.tolist()
    return top_features

def neutralize_predictions(df: pd.DataFrame, feature_cols: list, pred_col: str = 'prediction', proportion: float = 0.5) -> np.ndarray:
    """
    Orthogonalizes predictions against the selected features using linear algebra (least squares).
    This ensures that the model's predictions are purely additive alpha, eliminating exposure to 
    broader market factors that could cause severe underperformance during market regime shifts.
    """
    features = df[feature_cols].values
    # Add an intercept column to the feature matrix
    features = np.hstack((features, np.ones((len(features), 1))))
    preds = df[pred_col].values
    
    # Calculate exposure to features
    exposure = np.linalg.lstsq(features, preds, rcond=None)[0]
    
    # Calculate the linear correction
    correction = features.dot(exposure)
    
    # Subtract a proportion of the exposure from the original predictions
    neutralized_preds = preds - (proportion * correction)
    return neutralized_preds

## Base models (fixed hyperparameters — no per-call search)

Capped depth/leaves and a moderate `n_estimators` so a single fit is fast
at full data scale. These are reasonable starting points, not the result
of exhaustive tuning — see the offline tuning cell near the bottom if you
want to improve on them without paying that cost on every `train()` call.

In [ ]:
# @crunch/keep:on
XGB_FIXED_PARAMS = dict(
    n_estimators=256,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.7,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

LGBM_FIXED_PARAMS = dict(
    n_estimators=256,
    max_depth=6,
    num_leaves=63,
    learning_rate=0.03,
    subsample=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

RF_FIXED_PARAMS = dict(
    n_estimators=64,
    max_depth=6,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

SVR_FIXED_PARAMS = dict(
    C=1.0,
    epsilon=0.001,
    gamma='scale',
)

ETR_FIXED_PARAMS = dict(
    n_estimators=128,
    max_depth=12,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features='sqrt',
    random_state=RANDOM_STATE,
)
# @crunch/keep:off


def build_stack(X, y, groups, n_splits=3) -> StackingRegressor:
    """2-model stack + Ridge meta-learner. The internal out-of-fold
    predictions use precomputed moon-grouped splits, since
    StackingRegressor.fit() has no `groups` argument to forward to a
    GroupKFold splitter directly."""
    stack_cv = list(GroupKFold(n_splits=n_splits).split(X, y, groups=groups))
    return StackingRegressor(
        estimators=[
            ("xgb", xgb.XGBRegressor(**XGB_FIXED_PARAMS)),
            ("lgbm", LGBMRegressor(**LGBM_FIXED_PARAMS)),
            # ("rf", RandomForestRegressor(**RF_FIXED_PARAMS)),
            # ("svr", svm.SVR(**SVR_FIXED_PARAMS)),
            ("etr", ExtraTreesRegressor(**ETR_FIXED_PARAMS)),
        ],
        # final_estimator=Ridge(random_state=RANDOM_STATE),
        final_estimator=BayesianRidge(),
        cv=stack_cv,
        n_jobs=1,  # base learners already use n_jobs=-1 internally; avoid oversubscription
    )

## Train & infer

These two functions are the actual submission entry points, called
`train()` once per walk-forward step and `infer()` once per moon of live
data. Everything above is just setup they rely on.

In [71]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
) -> None:
    """Select features, fit the 2-model stack, and persist it for infer()."""

    feature_columns = get_feature_columns(X_train)
    X_full = X_train[feature_columns]
    y = y_train["target"].to_numpy()
    groups = X_train["moon"].to_numpy()  # group folds by week -> no time leakage
    
    selected_columns = select_features_by_correlation(X_full, y)
    X = X_full[selected_columns]

    stack = build_stack(X, y, groups)
    stack.fit(X, y)

    os.makedirs(model_directory_path, exist_ok=True)
    joblib.dump(
        {"model": stack, "feature_columns": selected_columns},
        get_model_path(model_directory_path),
    )


def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
) -> pd.DataFrame:
    """Load the persisted stack and score the current moon of data."""

    saved = joblib.load(get_model_path(model_directory_path))
    model, feature_columns = saved["model"], saved["feature_columns"]

    predictions = X_test[["id", "moon"]].copy()
    predictions["prediction"] = model.predict(X_test[feature_columns])
    
    return predictions

## Load the data

In [72]:
X_train, y_train, X_test = crunch_tools.load_data()

data/X.reduced.small.zip: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/X.reduced.small.zip (100411670 bytes)
data/X.reduced.small.zip: already exists, file length match
data/X.reduced.small.zip: already uncompressed, marker is present
data/moons_split.json: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/moons_split.json (11088 bytes)
data/moons_split.json: already exists, file length match
data/y.reduced.small.zip: download from https:crunchdao--competition--production.s3-accelerate.amazonaws.com/data-releases/210/y.reduced.small.zip (1158625 bytes)
data/y.reduced.small.zip: already exists, file length match
data/y.reduced.small.zip: already uncompressed, marker is present


## Held-out validation (last 10 moons) — and a timing check

This is your best local read on real runtime, though the local sample is
much smaller than the full dataset (13 moons for `X_test`/`y_test`, per
DataCrunch's docs, vs. hundreds in the real run) — treat the `%%time`
output here as a floor, not the true full-scale number.

In [73]:
%%time
feature_columns = get_feature_columns(X_train)
moons = np.sort(X_train["moon"].unique())
holdout_moons = moons[-10:]

is_holdout = X_train["moon"].isin(holdout_moons)
X_fit, y_fit = X_train.loc[~is_holdout], y_train.loc[~is_holdout]
X_val, y_val = X_train.loc[is_holdout], y_train.loc[is_holdout]

X_fit_full = X_fit[feature_columns]
y_fit_target = y_fit["target"].to_numpy()
groups_fit = X_fit["moon"].to_numpy()

selected_columns = select_features_by_correlation(X_fit_full, y_fit_target)
X_fit_sel = X_fit_full[selected_columns]

stack = build_stack(X_fit_sel, y_fit_target, groups_fit)
stack.fit(X_fit_sel, y_fit_target)

y_val_target = y_val["target"].to_numpy()
val_pred = stack.predict(X_val[selected_columns])

val_df = pd.DataFrame({'prediction': val_pred})
val_df = pd.concat([val_df, X_val[selected_columns].reset_index(drop=True)], axis=1)
neutralized_val_pred = neutralize_predictions(
    df=val_df, 
    feature_cols=selected_columns, 
    pred_col='prediction', 
    proportion=0.5
)

print(f"Holdout Spearman over last {len(holdout_moons)} moons: {spearman(y_val_target, val_pred):.4f}")
print(f"Holdout Spearman (Neutralized) over last {len(holdout_moons)} moons: {spearman(y_val_target, neutralized_val_pred):.4f}")

for name, base_model in stack.named_estimators_.items():
    base_pred = base_model.predict(X_val[selected_columns])
    print(f"  {name} alone: {spearman(y_val_target, base_pred):.4f}")

Holdout Spearman over last 10 moons: 0.0002
Holdout Spearman (Neutralized) over last 10 moons: 0.0011
  xgb alone: 0.0160
  rf alone: -0.0034
  etr alone: 0.0007
CPU times: user 1min 21s, sys: 2.59 s, total: 1min 24s
Wall time: 22.9 s


## Optional: offline hyperparameter exploration

**Not used by `train()`/`infer()` — run this manually if you want to
explore better hyperparameters, then copy whatever you find into
`XGB_FIXED_PARAMS`/`LGBM_FIXED_PARAMS` above.** Keeping search out of the
submitted `train()` is what keeps each of the 9 walk-forward calls fast;
this cell exists so you don't lose the ability to tune, just the cost of
doing it on every call. Consider running it against a row subsample if
even one search feels slow — it doesn't need the full dataset to point you
toward reasonable hyperparameters.

### Coarse search

In [ ]:
%%time
_search_params = {
    # XGBoost params ('xgb')
    "xgb__n_estimators": randint(32, 1024),
    "xgb__max_depth": randint(3, 18),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.06, 0.6), # Note: uniform(loc, scale) -> spans from loc to loc+scale
    "xgb__colsample_bytree": uniform(0.06, 0.6),

    # LGBM params ('lgbm')
    "lgbm__n_estimators": randint(32, 1024),
    "lgbm__max_depth": randint(3, 18),
    "lgbm__learning_rate": uniform(0.01, 0.15),
    "lgbm__subsample": uniform(0.06, 0.8), # Note: uniform(loc, scale) -> spans from loc to loc+scale
    
    # Random Forest params ('rf')
    # "rf__max_depth": randint(3, 12),
    # "rf__n_estimators": randint(50, 1024),
    
    # Extra Trees params ('etr')
    "etr__max_depth": randint(5, 15),
    "etr__n_estimators": randint(32, 1024),
}

_cv = GroupKFold(n_splits=3)

_search = RandomizedSearchCV(
    estimator=stack,
    param_distributions=_search_params,
    n_iter=200,
    cv=_cv,
    scoring=spearman_scorer,
    n_jobs=1,  # xgb already uses n_jobs=-1 internally
    random_state=RANDOM_STATE,
    verbose=1,
)
_search.fit(X_fit_sel, y_fit_target, groups=groups_fit)
with open("best_params_coarse.txt", "a") as f:
    print("Best stack params found:", _search.best_params_, file=f)
    print(f"Best CV Spearman: {_search.best_score_:.4f}", file=f)


Fitting 3 folds for each of 200 candidates, totalling 600 fits
CPU times: user 17min 52s, sys: 10.9 s, total: 18min 3s
Wall time: 2min 12s


KeyboardInterrupt: 

### Fine search

In [ ]:
bp = _search.best_params_

_fine_params = {
    # XGBoost params ('xgb')
    "xgb__n_estimators": randint(
        max(32, int(bp["xgb__n_estimators"] * 0.60)),
        int(bp["xgb__n_estimators"] * 1.40) + 1,
    ),
    "xgb__max_depth": randint(
        max(3, int(bp["xgb__max_depth"]) - 3),
        min(15, int(bp["xgb__max_depth"]) + 4),
    ),
    "xgb__learning_rate": uniform(
        max(0.005, bp["xgb__learning_rate"] - 0.05),
        min(0.10, bp["xgb__learning_rate"] + 0.05)
        - max(0.005, bp["xgb__learning_rate"] - 0.05),
    ),
    "xgb__subsample": uniform(
        max(0.40, bp["xgb__subsample"] - 0.20),
        min(1.00, bp["xgb__subsample"] + 0.20)
        - max(0.40, bp["xgb__subsample"] - 0.20),
    ),
    "xgb__colsample_bytree": uniform(
        max(0.40, bp["xgb__colsample_bytree"] - 0.20),
        min(1.00, bp["xgb__colsample_bytree"] + 0.20)
        - max(0.40, bp["xgb__colsample_bytree"] - 0.20),
    ),

    # LGBM params ('lgbm')
    "lgbm__n_estimators": randint(
        max(32, int(bp["lgbm__n_estimators"] * 0.60)),
        int(bp["lgbm__n_estimators"] * 1.40) + 1,
    ),
    "lgbm__max_depth": randint(
        max(3, int(bp["lgbm__max_depth"]) - 3),
        min(15, int(bp["lgbm__max_depth"]) + 4),
    ),
    "lgbm__learning_rate": uniform(
        max(0.005, bp["lgbm__learning_rate"] - 0.05),
        min(0.10, bp["lgbm__learning_rate"] + 0.05)
        - max(0.005, bp["lgbm__learning_rate"] - 0.05),
    ),
    "lgbm__subsample": uniform(
        max(0.40, bp["lgbm__subsample"] - 0.20),
        min(1.00, bp["lgbm__subsample"] + 0.20)
        - max(0.40, bp["lgbm__subsample"] - 0.20),
    ),

    # Random Forest params ('rf')
    # "rf__max_depth": randint(
    #     max(3, int(bp["rf__max_depth"]) - 3),
    #     min(20, int(bp["rf__max_depth"]) + 4),
    # ),
    # "rf__n_estimators": randint(
    #     max(32, int(bp["rf__n_estimators"] * 0.60)),
    #     int(bp["rf__n_estimators"] * 1.40) + 1,
    # ),

    # Extra Trees params ('etr')
    "etr__max_depth": randint(
        max(4, int(bp["etr__max_depth"]) - 3),
        min(22, int(bp["etr__max_depth"]) + 4),
    ),
    "etr__n_estimators": randint(
        max(50, int(bp["etr__n_estimators"] * 0.60)),
        int(bp["etr__n_estimators"] * 1.40) + 1,
    ),
}


_fine_cv = GroupKFold(n_splits=5)

_fine_search = RandomizedSearchCV(
    estimator=stack,
    param_distributions=_fine_params,
    n_iter=300,
    cv=_fine_cv,
    scoring=spearman_scorer,
    n_jobs=1,
    random_state=RANDOM_STATE + 1,
    verbose=1,
    return_train_score=False,
)

_fine_search.fit(
    X_fit_sel,
    y_fit_target,
    groups=groups_fit,
)

_fine_results = (
    pd.DataFrame(_fine_search.cv_results_)
    .sort_values("rank_test_score")
)

print("\nTop 10 fine-search configurations:")

print(
    _fine_results[
        [
            "rank_test_score",
            "mean_test_score",
            "std_test_score",
            "params",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

with open("best_params_fine.txt", "a") as f:
    print("Best stack params found:", _fine_search.best_params_, file=f)
    print(f"Best CV Spearman: {_fine_search.best_score_:.4f}", file=f)

AttributeError: 'RandomizedSearchCV' object has no attribute 'best_params_'

## Local test

`crunch_tools.test()` runs your `train()` / `infer()` exactly the way the
platform will, and checks the output format before you submit. Always run
this before pushing.

In [ ]:
crunch_tools.test()